# CottonDoctor-PK — training
Cotton leaf disease classifier. Transfer learning on EfficientNet-B0 over the raw split of `Project-AgML/cotton_leaf_disease_classification_bangladesh_2` (CC-BY-4.0).

In [ ]:
!pip install -q pyarrow
import io, json, os, random, shutil
from collections import Counter
import numpy as np
import pyarrow.parquet as pq
import torch, torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
CLASSES = ['Bacterial Blight', 'Curl Virus', 'Healthy Leaf', 'Herbicide Growth Damage',
           'Leaf Hopper Jassids', 'Leaf Redding', 'Leaf Variegation']
os.makedirs('data', exist_ok=True)

In [ ]:
!curl -sL -o raw-train.parquet "https://huggingface.co/datasets/Project-AgML/cotton_leaf_disease_classification_bangladesh_2/resolve/main/raw/train-00000-of-00001.parquet"
t = pq.read_table('raw-train.parquet')
labels, imgs = t.column('label').to_pylist(), t.column('image').to_pylist()
raw = 'data/raw'
for i, (lab, img) in enumerate(zip(labels, imgs)):
    d = os.path.join(raw, str(lab)); os.makedirs(d, exist_ok=True)
    Image.open(io.BytesIO(img['bytes'])).convert('RGB').resize((256, 256)).save(os.path.join(d, f'{i}.jpg'), quality=90)
print('class counts:', dict(sorted(Counter(labels).items())))

In [ ]:
items = [(c, f) for c in os.listdir(raw) for f in os.listdir(os.path.join(raw, c))]
y = [int(c) for c, _ in items]
tr, te = train_test_split(items, test_size=0.15, random_state=SEED, stratify=y)
tr, va = train_test_split(tr, test_size=0.1765, random_state=SEED, stratify=[int(c) for c, _ in tr])
for name, part in [('train', tr), ('val', va), ('test', te)]:
    for c, f in part:
        d = os.path.join('data', name, c); os.makedirs(d, exist_ok=True)
        shutil.copy(os.path.join(raw, c, f), os.path.join(d, f))
print({n: sum(len(fs) for _, _, fs in os.walk(os.path.join('data', n))) for n in ['train', 'val', 'test']})

In [ ]:
def load(split, size=64):
    xs, ys = [], []
    for c in sorted(os.listdir(os.path.join('data', split))):
        for f in os.listdir(os.path.join('data', split, c)):
            im = Image.open(os.path.join('data', split, c, f)).convert('RGB').resize((size, size))
            xs.append(np.asarray(im)); ys.append(int(c))
    return np.array(xs), np.array(ys)

xtr, ytr = load('train'); xte, yte = load('test')
maj = Counter(ytr).most_common(1)[0][0]
print('majority baseline:', round(float(accuracy_score(yte, np.full_like(yte, maj))), 4))
hist = lambda x: np.stack([np.histogramdd(p.reshape(-1, 3), bins=8, range=((0, 256),) * 3)[0].ravel() for p in x])
sc = StandardScaler().fit(hist(xtr))
clf = LogisticRegression(max_iter=2000).fit(sc.transform(hist(xtr)), ytr)
print('hist+logreg baseline:', round(float(accuracy_score(yte, clf.predict(sc.transform(hist(xte))))), 4)

In [ ]:
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tr_tf = transforms.Compose([transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
ev_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), norm])

class LeafDS(Dataset):
    def __init__(self, split, tf):
        self.items = [(os.path.join('data', split, c, f), int(c))
                        for c in sorted(os.listdir(os.path.join('data', split)))
                        for f in os.listdir(os.path.join('data', split, c))]
        self.tf = tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), y

tr = DataLoader(LeafDS('train', tr_tf), batch_size=32, shuffle=True, num_workers=2)
va = DataLoader(LeafDS('val', ev_tf), batch_size=64, num_workers=2)
te = DataLoader(LeafDS('test', ev_tf), batch_size=64, num_workers=2)

net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
net.classifier[1] = nn.Linear(net.classifier[1].in_features, 7)
net.to(DEVICE)
counts = Counter(y for _, y in LeafDS('train', ev_tf))
w = torch.tensor([len(tr.dataset) / counts[c] for c in range(7)], dtype=torch.float32).to(DEVICE)
crit = nn.CrossEntropyLoss(weight=w)

def run(epochs, lr, params):
    opt = torch.optim.Adam(params, lr=lr)
    best, best_state, wait = 0, None, 0
    for ep in range(epochs):
        net.train()
        for x, y in tr:
            opt.zero_grad(); crit(net(x.to(DEVICE)), y.to(DEVICE)).backward(); opt.step()
        net.eval(); correct, total = 0, 0
        with torch.no_grad():
            for x, y in va:
                p = net(x.to(DEVICE)).argmax(1).cpu()
                correct += (p == y).sum().item(); total += len(y)
        acc = correct / total
        print(f'epoch {ep+1}/{epochs} val_acc {acc:.4f}')
        if acc > best: best, best_state, wait = acc, {k: v.cpu().clone() for k, v in net.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 4: break
    net.load_state_dict(best_state)

for p in net.features.parameters(): p.requires_grad = False
run(15, 1e-3, net.classifier.parameters())
for p in list(net.features.parameters())[-20:]: p.requires_grad = True
run(6, 1e-4, [p for p in net.parameters() if p.requires_grad])

In [ ]:
net.eval(); allp, ally = [], []
with torch.no_grad():
    for x, y in te:
        allp += net(x.to(DEVICE)).argmax(1).cpu().tolist(); ally += y.tolist()
print('TEST ACCURACY:', round(float(accuracy_score(ally, allp)), 4))
print(classification_report(ally, allp, target_names=CLASSES, zero_division=0))
print(confusion_matrix(ally, allp))
torch.save(net.state_dict(), 'model.pt')
print('saved model.pt:', os.path.getsize('model.pt'), 'bytes')